# CAP4D Static Avatar for Unity (Colab)
This notebook runs tracking + generation + avatar fitting and exports a static Unity-friendly 3DGS `.ply`.

In [ ]:
# Top-level settings
QUALITY = "balanced"   # "balanced" | "max" | "debug"
MAX_N_REF = 48           # reduce for speed, increase for quality
TIMESTEP = 0             # static FLAME timestep to bake
INPUT_VIDEO_PATH = "/content/my_head_video.mp4"
OUTPUT_PATH = "/content/cap4d/examples/output/custom_static"
REPO_URL = "https://github.com/vikram-menon/cap4d.git"  # set to your fork if needed
REPO_REF = "colab"  # branch/tag/commit containing static export scripts

In [ ]:
!nvidia-smi -L

In [ ]:
import subprocess
%cd /content
!rm -rf cap4d
subprocess.run(["git", "clone", REPO_URL, "cap4d"], check=True)
%cd /content/cap4d
subprocess.run(["git", "checkout", REPO_REF], check=True)

In [ ]:
import os
os.environ["CAP4D_PATH"] = "/content/cap4d"
os.environ["PIXEL3DMM_PATH"] = "/content/pixel3dmm"
os.environ["PYTHONPATH"] = f"/content/cap4d:{os.environ.get('PYTHONPATH', '')}"
print('CAP4D_PATH=', os.environ['CAP4D_PATH'])
print('PIXEL3DMM_PATH=', os.environ['PIXEL3DMM_PATH'])

In [ ]:
%%bash
set -e
cd /content/cap4d
pip install -r requirements.txt
export FORCE_CUDA=1
pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable"
apt-get update
apt-get install -y ffmpeg

In [ ]:
import os
import getpass
os.environ['FLAME_USERNAME'] = input('FLAME username: ')
os.environ['FLAME_PWD'] = getpass.getpass('FLAME password: ')

In [ ]:
%%bash
set -e
cd /content/cap4d
bash scripts/download_flame.sh
bash scripts/download_mmdm_weights.sh
bash scripts/install_pixel3dmm.sh

In [ ]:
# Patch Pixel3DMM for torch>=2.6 checkpoint loading + facer indexing
from pathlib import Path

base = Path('/content/pixel3dmm/scripts')

# 1) torch>=2.6 changed torch.load default to weights_only=True
p = base / 'network_inference.py'
s = p.read_text()
old = 'load_from_checkpoint(model_checkpoint, strict=False)'
new = 'load_from_checkpoint(model_checkpoint, strict=False, weights_only=False)'
if old in s:
    p.write_text(s.replace(old, new))
    print('Patched:', p, '(weights_only=False)')
else:
    print('Already patched or pattern not found:', p)

# 2) Prevent IndexError in facer segmentation when image_ids exceed local batch size
p = base / 'run_facer_segmentation.py'
s = p.read_text()
needle = 'frame = frame_idx_batch[_iidx]'
replacement = 'idx = int(_iidx)\n            if idx < 0 or idx >= len(frame_idx_batch):\n                continue\n            frame = frame_idx_batch[idx]'
if needle in s:
    p.write_text(s.replace(needle, replacement))
    print('Patched:', p, '(frame_idx bounds check)')
else:
    print('Already patched or pattern not found:', p)


In [ ]:
# Optional upload: use this cell if INPUT_VIDEO_PATH is not already available in /content
from google.colab import files
uploaded = files.upload()
if uploaded:
    INPUT_VIDEO_PATH = f"/content/{next(iter(uploaded.keys()))}"
print('INPUT_VIDEO_PATH =', INPUT_VIDEO_PATH)

In [ ]:
import os
import re
import shlex
import subprocess
import threading
import time
from datetime import datetime
from pathlib import Path
LOG_DIR = "/content/cap4d_logs"
os.makedirs(LOG_DIR, exist_ok=True)

# Rough ranges to give a practical expectation. These are not strict predictions.
ETA_HINTS_MIN = {
    "setup_clone": 1,
    "install_core": 8,
    "download_weights": 3,
    "install_pixel3dmm": 10,
    "tracking": 20,
    "generate_images": 30,
    "train_avatar": 45,
    "export_static": 1,
}
ETA_HINTS_MAX = {
    "setup_clone": 3,
    "install_core": 25,
    "download_weights": 10,
    "install_pixel3dmm": 35,
    "tracking": 90,
    "generate_images": 180,
    "train_avatar": 360,
    "export_static": 5,
}


def _now():
    return datetime.now().strftime("%H:%M:%S")


def run_logged(name, cmd, cwd=None, env=None, shell=False):
    """Run command with streamed logs, heartbeat, logfile, and rough ETA range."""
    log_path = Path(LOG_DIR) / f"{name}.log"
    start = time.time()
    last_line_time = [start]

    eta_min = ETA_HINTS_MIN.get(name)
    eta_max = ETA_HINTS_MAX.get(name)
    if eta_min is not None and eta_max is not None:
        print(f"[{_now()}] [{name}] ETA (rough): {eta_min}-{eta_max} min")

    print(f"[{_now()}] [{name}] START")
    print(f"[{_now()}] [{name}] CMD: {cmd if isinstance(cmd, str) else ' '.join(shlex.quote(c) for c in cmd)}")
    print(f"[{_now()}] [{name}] LOG: {log_path}")

    if shell:
        popen_cmd = cmd
    else:
        popen_cmd = cmd if isinstance(cmd, list) else shlex.split(cmd)

    process = subprocess.Popen(
        popen_cmd,
        cwd=cwd,
        env=env,
        shell=shell,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    stop_flag = {"stop": False}

    def heartbeat():
        while not stop_flag["stop"]:
            time.sleep(30)
            if stop_flag["stop"]:
                break
            elapsed = (time.time() - start) / 60.0
            silent = time.time() - last_line_time[0]
            msg = f"[{_now()}] [{name}] heartbeat: elapsed={elapsed:.1f}m"
            if silent >= 30:
                msg += f", no new output for {silent:.0f}s"
            if eta_max is not None:
                remaining_floor = max(0.0, eta_max - elapsed)
                msg += f", est remaining <= {remaining_floor:.1f}m"
            print(msg)

    t = threading.Thread(target=heartbeat, daemon=True)
    t.start()

    with log_path.open("w", encoding="utf-8") as f:
        for raw in process.stdout:
            line = raw.rstrip("\n")
            last_line_time[0] = time.time()
            elapsed = time.time() - start
            prefix = f"[{_now()}] [{name}] [+{elapsed:7.1f}s]"
            print(f"{prefix} {line}")
            f.write(raw)

    rc = process.wait()
    stop_flag["stop"] = True
    total = (time.time() - start) / 60.0

    if rc != 0:
        print(f"[{_now()}] [{name}] FAIL rc={rc} after {total:.1f}m")
        print(f"[{_now()}] [{name}] See log: {log_path}")
        raise RuntimeError(f"Step '{name}' failed (rc={rc}). Log: {log_path}")

    print(f"[{_now()}] [{name}] DONE in {total:.1f}m")
    return str(log_path)


print("Logging helper loaded. Logs will be written under:", LOG_DIR)


In [ ]:
# Run full static pipeline in one command with logs
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

run_logged(
    name="tracking",
    cmd=[
        "bash", "scripts/track_video_pixel3dmm.sh",
        INPUT_VIDEO_PATH,
        f"{OUTPUT_PATH}/reference_tracking",
        "--max_n_ref", str(MAX_N_REF),
    ],
    cwd="/content/cap4d",
)

run_logged(
    name="generate_images",
    cmd=[
        "python", "cap4d/inference/generate_images.py",
        "--config_path", "configs/generation/low_quality.yaml" if QUALITY == "balanced" else ("configs/generation/high_quality.yaml" if QUALITY == "max" else "configs/generation/debug.yaml"),
        "--reference_data_path", f"{OUTPUT_PATH}/reference_tracking",
        "--output_path", f"{OUTPUT_PATH}/mmdm",
    ],
    cwd="/content/cap4d",
)

run_logged(
    name="train_avatar",
    cmd=[
        "python", "gaussianavatars/train.py",
        "--config_path", "configs/avatar/low_quality.yaml" if QUALITY == "balanced" else ("configs/avatar/high_quality.yaml" if QUALITY == "max" else "configs/avatar/debug.yaml"),
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
    ],
    cwd="/content/cap4d",
)

run_logged(
    name="export_static",
    cmd=[
        "python", "gaussianavatars/export_static_ply.py",
        "--model_path", f"{OUTPUT_PATH}/avatar/",
        "--source_paths", f"{OUTPUT_PATH}/mmdm/reference_images/", f"{OUTPUT_PATH}/mmdm/generated_images/",
        "--output_ply", f"{OUTPUT_PATH}/raw_static.ply",
        "--timestep", str(TIMESTEP),
    ],
    cwd="/content/cap4d",
)


In [ ]:
from google.colab import files
ply_path = f"{OUTPUT_PATH}/raw_static.ply"
print('Exported PLY:', ply_path)
files.download(ply_path)